In [1]:
import pyreadstat
import pandas as pd
import numpy as np
from pathlib import Path
import os

os.chdir("/home/isaac/Documents/BigData/Almacenes_de_Datos/Proyecto/canasta-basica-cr")

BASE    = Path("data/raw")
OUTPUT  = Path("data/processed")
OUTPUT.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════
# BLOQUE 1 — ENAHO: consolidar 11 años en una sola tabla
# ════════════════════════════════════════════════════════

COLS_ENAHO = [
    "FACTOR", "ID_HOGAR", "REGION", "ZONA", "TamHog",
    "A4", "A5", "NivInst", "CondAct",
    "ithb", "ithn", "ipcb", "ipcn",
    "cba", "lp", "np",
    "Q_IPCN", "D_IPCN", "IPM_Pobreza",
]

ARCHIVOS_ENAHO = {
    2010: BASE / "enaho/ENAHO 2010.sav",
    2011: BASE / "enaho/ENAHO 2011.sav",
    2012: BASE / "enaho/ENAHO 2012.sav",
    2015: BASE / "enaho/ENAHO 2015.sav",
    2018: BASE / "enaho/ENAHO 2018.sav",
    2020: BASE / "enaho/ENAHO 2020.sav",
    2021: BASE / "enaho/ENAHO 2021.sav",
    2022: BASE / "enaho/BdBasePublica 2022.sav",
    2023: BASE / "enaho/GPES-ELAB-GEBD-ENAHO-2023_BdBasePublica.sav",
    2024: BASE / "enaho/GPES-ELAB-GEBD-ENAHO-2024_BdPublica.sav",
    2025: BASE / "enaho/GPES-ELAB-GEBD-ENAHO-2025_BdPublica.sav",
}

bloques = []

for anio, ruta in ARCHIVOS_ENAHO.items():
    print(f"  Leyendo Enaho {anio}...", end=" ")

    # Leer solo columnas que existen en ese año
    _, meta = pyreadstat.read_sav(str(ruta), row_limit=0)
    cols_disponibles = [c for c in COLS_ENAHO if c in meta.column_names]
    cols_faltantes   = [c for c in COLS_ENAHO if c not in meta.column_names]

    df, _ = pyreadstat.read_sav(str(ruta), usecols=cols_disponibles)

    # Agregar columnas faltantes como NaN (años anteriores pueden no tenerlas)
    for col in cols_faltantes:
        df[col] = np.nan

    df["anio"] = anio   # columna de año para trazabilidad

    bloques.append(df)
    print(f"{len(df):,} filas | cols faltantes: {cols_faltantes if cols_faltantes else 'ninguna'}")

df_enaho = pd.concat(bloques, ignore_index=True)
print(f"\n✅ Enaho consolidada: {len(df_enaho):,} filas × {len(df_enaho.columns)} columnas")

# ── Limpieza básica ──────────────────────────────────────
# Renombrar a nombres descriptivos
df_enaho = df_enaho.rename(columns={
    "A4":       "sexo",           # 1=hombre, 2=mujer
    "A5":       "edad",
    "NivInst":  "nivel_instruccion",
    "CondAct":  "condicion_actividad",
    "TamHog":   "tam_hogar",
    "REGION":   "region",
    "ZONA":     "zona",           # 1=urbano, 2=rural
    "ithb":     "ingreso_hogar_bruto",
    "ipcb":     "ingreso_percapita_bruto",
    "cba":      "valor_cba",      # TARGET del modelo
    "lp":       "linea_pobreza",
    "np":       "nivel_pobreza",  # 1=no pobre, 2=pobreza básica, 3=pobreza extrema
    "Q_IPCN":   "quintil_ingreso",
    "D_IPCN":   "decil_ingreso",
    "IPM_Pobreza": "pobreza_multidimensional",
})

# Imputar nulos en ingreso con mediana por región y año
for col in ["ingreso_hogar_bruto", "ingreso_percapita_bruto"]:
    mediana = df_enaho.groupby(["anio", "region"])[col].transform("median")
    df_enaho[col] = df_enaho[col].fillna(mediana)

# Guardar
df_enaho.to_parquet(OUTPUT / "enaho_consolidada.parquet", index=False)
print(f"💾 Guardado: data/processed/enaho_consolidada.parquet")
print(f"   Años cubiertos: {sorted(df_enaho['anio'].unique())}")
print(f"   Distribución de nivel_pobreza:\n{df_enaho['nivel_pobreza'].value_counts(dropna=False)}")

  Leyendo Enaho 2010... 41,184 filas | cols faltantes: ['ID_HOGAR', 'REGION', 'ZONA', 'A4', 'A5', 'IPM_Pobreza']
  Leyendo Enaho 2011... 40,860 filas | cols faltantes: ['ID_HOGAR', 'IPM_Pobreza']
  Leyendo Enaho 2012... 39,390 filas | cols faltantes: ['ID_HOGAR', 'IPM_Pobreza']
  Leyendo Enaho 2015... 37,291 filas | cols faltantes: ninguna
  Leyendo Enaho 2018... 35,096 filas | cols faltantes: ninguna
  Leyendo Enaho 2020... 25,530 filas | cols faltantes: ninguna
  Leyendo Enaho 2021... 31,963 filas | cols faltantes: ninguna
  Leyendo Enaho 2022... 31,045 filas | cols faltantes: ninguna
  Leyendo Enaho 2023... 30,540 filas | cols faltantes: ninguna
  Leyendo Enaho 2024... 30,846 filas | cols faltantes: ninguna
  Leyendo Enaho 2025... 30,098 filas | cols faltantes: ninguna

✅ Enaho consolidada: 373,843 filas × 20 columnas


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.